# Business Entity Resolution — full pipeline
Needs: the challenge data added as an input dataset, **Internet ON** (for pip). CPU session for `MODEL = 'lgb'`, GPU session for `MODEL = 'xgb'`.
Outputs land in `/kaggle/working/output/`.

In [ ]:
!pip install -q polars==1.44.2 rapidfuzz==3.14.6 anyascii==0.3.3 sparse_dot_topn==1.2.0 lightgbm==4.7.0 xgboost==3.4.1
import os, platform, multiprocessing, importlib.metadata as md
os.makedirs('/kaggle/working/src', exist_ok=True); os.makedirs('/kaggle/working/output', exist_ok=True)
pkgs = ['anyascii', 'lightgbm', 'numpy', 'polars', 'pyarrow', 'rapidfuzz', 'scikit-learn', 'scipy', 'sparse_dot_topn', 'xgboost']
env = [f'python=={platform.python_version()}', f'cpus={multiprocessing.cpu_count()}']
env += [f'{p}=={md.version(p)}' for p in pkgs]
open('/kaggle/working/output/environment.txt', 'w').write('\n'.join(env) + '\n')
print('\n'.join(env))
%cd /kaggle/working
import subprocess
def run(cmd):
    """Run a pipeline step, stream its output, and stop the notebook if it fails."""
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    rc = p.wait()
    if rc:
        why = {-9: ' - KILLED, almost certainly out of RAM', 137: ' - KILLED, almost certainly out of RAM',
               -11: ' - SEGFAULT inside a library (e.g. XGBoost/LightGBM)', 139: ' - SEGFAULT inside a library'}.get(rc, '')
        raise RuntimeError(f'{cmd!r} failed with exit code {rc}{why}')

In [ ]:
%%writefile src/config.py
"""Paths and knobs. Override paths with env vars so the same code runs locally or on Kaggle."""
import os
from pathlib import Path

ROOT = Path(__file__).resolve().parents[3]
ON_KAGGLE = Path("/kaggle/input").exists()


def _find_kaggle_data():
    hits = [p for p in Path("/kaggle/input").rglob("train_source1.tsv") if "__MACOSX" not in str(p)]
    if not hits:
        raise FileNotFoundError("train_source1.tsv not found under /kaggle/input - add the dataset as input")
    return hits[0].parent.parent


DATA_DIR = Path(os.environ.get("ER_DATA_DIR") or (_find_kaggle_data() if ON_KAGGLE else ROOT / "student_resource" / "dataset"))
WORK_DIR = Path(os.environ.get("ER_WORK_DIR") or ("/kaggle/working/work" if ON_KAGGLE else ROOT / "work"))
OUT_DIR = Path(os.environ.get("ER_OUT_DIR") or ("/kaggle/working/output" if ON_KAGGLE else ROOT / "output"))
N_JOBS = int(os.environ.get("ER_N_JOBS", os.cpu_count() or 4))
SEED = 42

WORK_DIR.mkdir(parents=True, exist_ok=True)


def src_path(split, s):
    return DATA_DIR / split / f"{split}_source{s}.tsv"


# Bump when normalization changes: cached norm/cand files from older versions are then ignored.
NORM_VERSION = 3


def norm_path(split, s):
    return WORK_DIR / f"norm_v{NORM_VERSION}_{split}_s{s}.parquet"


def cand_path(split):
    return WORK_DIR / f"cand_v{NORM_VERSION}_{split}.parquet"


def translit_path():
    return WORK_DIR / f"translit_v{NORM_VERSION}.json"


In [ ]:
%%writefile src/normalize.py
"""Record normalization shared by blocking and feature extraction.

Everything here is rule-based and country-agnostic by default; country-specific
abbreviation tables are applied only when the country label matches, and unknown
countries fall back to the generic table.
"""
import re

from anyascii import anyascii

from translit import INDIC_RE

# Indic back-transliteration (translit.Translit), installed by prepare.py; None = disabled.
_TRANSLIT = None


def set_translit(tr):
    global _TRANSLIT
    _TRANSLIT = tr

# ---------------------------------------------------------------- legal suffixes
# Token -> legal-form class. Transliterated Indic forms (via anyascii) included.
LEGAL = {
    "llc": "llc", "inc": "inc", "incorporated": "inc", "corp": "corp", "corporation": "corp",
    "co": "co", "company": "co", "cie": "co", "ltd": "ltd", "limited": "ltd", "limitet": "ltd",
    "pvt": "pvt", "private": "pvt", "praivet": "pvt", "piraivet": "pvt", "pra": "pvt", "li": "ltd",
    "llp": "llp", "elelpi": "llp", "pc": "pc", "pllc": "pllc", "lp": "lp", "plc": "plc",
    "sarl": "sarl", "sas": "sas", "sasu": "sasu", "eurl": "eurl", "sa": "sa", "sci": "sci",
    "snc": "snc", "scp": "scp", "selarl": "selarl", "scop": "scop", "gmbh": "gmbh",
}
# Tokens dropped from the "core" name (legal forms + function words + web noise).
STOP = set(LEGAL) | {
    "the", "and", "of", "ms", "m", "s", "www", "com", "net", "org", "in", "fr", "de", "du", "des",
    "la", "le", "les", "et", "a", "an", "l", "d", "li",
}

DOMAIN_RE = re.compile(r"\b(?:https?://)?(?:www\.)?([a-z0-9\-]{3,})\.(?:com|net|org|co\.in|in|fr|biz|info|us)\b")
ID_JUNK_RE = re.compile(r"\(\s*id\s*:?\s*\d+\s*\)|#\s*\d+|\bid\s*:\s*\d+")
DOT_IN_WORD_RE = re.compile(r"(?<=[a-z])\.(?=[a-z])")
NON_ALNUM_RE = re.compile(r"[^a-z0-9]+")
DIGITS_RE = re.compile(r"\d+")

# ---------------------------------------------------------------- addresses
ADDR_GENERIC = {
    "st": "street", "str": "street", "rd": "road", "ave": "avenue", "av": "avenue", "blvd": "boulevard",
    "dr": "drive", "ln": "lane", "ct": "court", "cir": "circle", "pl": "place", "pkwy": "parkway",
    "hwy": "highway", "trl": "trail", "ter": "terrace", "sq": "square", "mt": "mount", "ft": "fort",
    "apt": "apartment", "ste": "suite", "fl": "floor", "flr": "floor", "bldg": "building", "bldng": "building",
    "nr": "near", "opp": "opposite", "mkt": "market", "sec": "sector", "ph": "phase", "dist": "district",
    "n": "north", "s": "south", "e": "east", "w": "west", "hno": "", "no": "", "unit": "", "po": "",
    "null": "", "none": "", "na": "", "nan": "",
}
ADDR_BY_COUNTRY = {
    "france": {**ADDR_GENERIC, "r": "rue", "bd": "boulevard", "q": "quai", "imp": "impasse",
               "all": "allee", "che": "chemin", "chem": "chemin", "rte": "route", "fbg": "faubourg",
               "st": "saint", "ste": "sainte", "pl": "place", "av": "avenue", "bis": "bis", "ter": "ter",
               "n": "", "e": "", "s": ""},
}
US_STATES = {
    "alabama": "al", "alaska": "ak", "arizona": "az", "arkansas": "ar", "california": "ca", "colorado": "co",
    "connecticut": "ct", "delaware": "de", "florida": "fl", "georgia": "ga", "hawaii": "hi", "idaho": "id",
    "illinois": "il", "indiana": "in", "iowa": "ia", "kansas": "ks", "kentucky": "ky", "louisiana": "la",
    "maine": "me", "maryland": "md", "massachusetts": "ma", "michigan": "mi", "minnesota": "mn",
    "mississippi": "ms", "missouri": "mo", "montana": "mt", "nebraska": "ne", "nevada": "nv",
    "new hampshire": "nh", "new jersey": "nj", "new mexico": "nm", "new york": "ny", "north carolina": "nc",
    "north dakota": "nd", "ohio": "oh", "oklahoma": "ok", "oregon": "or", "pennsylvania": "pa",
    "rhode island": "ri", "south carolina": "sc", "south dakota": "sd", "tennessee": "tn", "texas": "tx",
    "utah": "ut", "vermont": "vt", "virginia": "va", "washington": "wa", "west virginia": "wv",
    "wisconsin": "wi", "wyoming": "wy", "district of columbia": "dc",
}
_US_STATE_RE = re.compile(r"\b(" + "|".join(sorted(US_STATES, key=len, reverse=True)) + r")\s*$")


def _ascii_lower(s):
    if not s:
        return ""
    return anyascii(s).lower()


def norm_name(raw):
    """Returns dict with normalized name pieces."""
    s = _ascii_lower(raw)
    domains = DOMAIN_RE.findall(s)
    s = DOMAIN_RE.sub(lambda m: " " + m.group(1).replace("-", " ") + " ", s)
    s = ID_JUNK_RE.sub(" ", s).replace("&", " and ").replace("m/s", " ")
    s = DOT_IN_WORD_RE.sub("", s)  # l.l.c -> llc, s.a.s -> sas
    toks = NON_ALNUM_RE.sub(" ", s).split()
    if _TRANSLIT is not None and raw and INDIC_RE.search(raw):
        toks = _TRANSLIT.map_tokens(toks)
    legal = sorted({LEGAL[t] for t in toks if t in LEGAL})
    core, prev = [], None
    for t in toks:
        if t in STOP or t == prev:
            continue
        core.append(t)
        prev = t
    return {
        "name_n": " ".join(toks),
        "core": " ".join(core),
        "compact": "".join(core),
        "legal": "|".join(legal),
        "is_domain": bool(domains),
        "name_nonascii": bool(raw) and not raw.isascii(),
        "name_nums": " ".join(str(int(x)) for x in DIGITS_RE.findall(" ".join(core))),
    }


def norm_addr(raw, country):
    s = _ascii_lower(raw)
    c = (country or "").strip().lower()
    table = ADDR_BY_COUNTRY.get(c, ADDR_GENERIC)
    if c in ("us", "usa", "united states"):
        s = _US_STATE_RE.sub(lambda m: US_STATES[m.group(1)], s.strip())
    nums = [str(int(x)) for x in DIGITS_RE.findall(s)]
    toks = []
    for t in NON_ALNUM_RE.sub(" ", s.replace("n°", " ")).split():
        if t.isdigit():
            continue
        m = re.fullmatch(r"(\d+)(st|nd|rd|th|bis|ter)?", t)
        if m:
            continue
        t = re.sub(r"\d+", "", t) if not t.isalpha() else t   # 25th / 3rt -> th / rt noise
        t = table.get(t, t)
        if len(t) >= 2:
            toks.append(t)
    missing = not toks and not nums
    return {"addr_n": " ".join(toks), "anums": " ".join(nums), "addr_missing": missing}


def normalize_record(eid, name, addr, country):
    out = {"entity_id": eid, "country": country}
    out.update(norm_name(name))
    out.update(norm_addr(addr, country))
    return out


In [ ]:
%%writefile src/prepare.py
"""Stage 1: learn the Indic back-transliteration table from train, then read raw TSVs,
normalize every record in parallel and cache as parquet.

    python prepare.py train test
"""
import collections
import sys
import time
from multiprocessing import Pool

import polars as pl

import normalize
import translit
from config import DATA_DIR, N_JOBS, norm_path, src_path, translit_path
from normalize import normalize_record

CHUNK = 20000


def read_tsv(path):
    return pl.read_csv(path, separator="\t", quote_char=None, infer_schema=False)


def _tokens(name):
    return normalize.norm_name(name)["name_n"].split()   # called with transliteration disabled


def build_translit():
    """Aligned (S1 words, romanized S2/S3 words) from train true pairs whose S2/S3 name is in an
    Indic script, plus the S1 name vocabulary of train and test (unlabeled) for the fallback."""
    t = time.time()
    gt = (read_tsv(DATA_DIR / "train" / "train_ground_truth.tsv")
          .with_columns(pl.col("matched_entity_ids").fill_null("").str.split(","))
          .explode("matched_entity_ids").rename({"source1_entity_id": "s1", "matched_entity_ids": "o"}))
    s1 = read_tsv(src_path("train", 1)).select(pl.col("entity_id").alias("s1"), pl.col("business_name").alias("n1"))
    pairs = []
    for s in (2, 3):
        o = (read_tsv(src_path("train", s)).select("entity_id", "business_name")
             .filter(pl.col("business_name").str.contains(r"[ऀ-෿]")))
        pairs.append(o.join(gt, left_on="entity_id", right_on="o").join(s1, on="s1")
                     .select("n1", pl.col("business_name").alias("no")))
    pairs = pl.concat(pairs)
    vocab = collections.Counter()
    for split in ("train", "test"):
        names = read_tsv(src_path(split, 1)).filter(pl.col("country") == "India")["business_name"]
        vocab.update(tok for n in names.to_list() for tok in _tokens(n))
    with Pool(N_JOBS) as pool:
        a = pool.map(_tokens, pairs["n1"].to_list(), chunksize=5000)
        b = pool.map(_tokens, pairs["no"].to_list(), chunksize=5000)
    table = translit.build(zip(a, b), vocab)
    translit.save(table, translit_path())
    print(f"translit: {pairs.height:,} Indic pairs -> {len(table['learned']):,} learned words, "
          f"{len(table['fallback']):,} fallback skeletons ({time.time() - t:.0f}s)", flush=True)


def _init_worker(path):
    normalize.set_translit(translit.load(path))


def _work(rows):
    return [normalize_record(*r) for r in rows]


def prepare(split):
    for s in (1, 2, 3):
        t = time.time()
        df = read_tsv(src_path(split, s))
        rows = list(zip(df["entity_id"], df["business_name"], df["business_address"], df["country"]))
        del df
        chunks = [rows[i:i + CHUNK] for i in range(0, len(rows), CHUNK)]
        out = []
        with Pool(N_JOBS, initializer=_init_worker, initargs=(str(translit_path()),)) as pool:
            for part in pool.imap(_work, chunks, chunksize=1):
                out.append(pl.DataFrame(part))
        del rows, chunks
        res = pl.concat(out).with_columns(pl.col("country").fill_null(""))
        res.write_parquet(norm_path(split, s))
        print(f"{split} s{s}: {res.height:,} rows in {time.time() - t:.0f}s", flush=True)


if __name__ == "__main__":
    if not translit_path().exists():
        build_translit()
    for split in sys.argv[1:] or ["train", "test"]:
        prepare(split)


In [ ]:
%%writefile src/blocking.py
"""Stage 2: candidate generation (three channels, unioned).

For every country label (open set) each record is hashed into a sparse bag of blocking
features, IDF-weighted on that country's S1+S2+S3 records (features that are too common or
appear only once are dropped) and L2-normalized. Three searches then run per source:

  * forward  : top-K_FWD S2/S3 records per S1 on the full name+address vector
  * name     : top-K_NAME S2/S3 records per S1 on the name-only vector
               (recovers same-name records whose address is missing or very different)
  * reverse  : top-K_REV S1 records per S2/S3 record on the full vector
               (recovers true owners pushed out of an S1's forward list by lookalikes)

The union is written with both cosine scores and the rank in each channel (99 = not found).

    python blocking.py train
"""
import sys
import time
from multiprocessing import Pool

import numpy as np
import polars as pl
import scipy.sparse as sp
from sklearn.feature_extraction.text import HashingVectorizer
from sparse_dot_topn import sp_matmul_topn

from config import N_JOBS, cand_path, norm_path

N_FEATURES = 2 ** 23
MAX_DF = 3000          # features present in more records than this (per country) are dropped
K_FWD, K_NAME, K_REV = 20, 0, 5   # K_NAME = 0 disables the name-only channel (v2: +0.001 oracle F0.5 for +5 cands/S1)
NOT_FOUND = 99
COLS = ["core", "compact", "addr_n", "anums", "name_nums"]


def name_feats(core, compact, name_nums):
    feats = ["w:" + t for t in core.split()]
    c = f"^{compact}$"
    feats += ["c:" + c[i:i + 4] for i in range(max(1, len(c) - 3))]
    feats += ["m:" + n for n in name_nums.split()]
    return feats


def analyzer(doc):
    core, compact, addr, anums, name_nums = doc
    feats = name_feats(core, compact, name_nums)
    words = [t for t in addr.split() if len(t) >= 3]
    feats += ["a:" + t for t in words]
    nums = anums.split()[:4]
    feats += ["n:" + n for n in nums if len(n) >= 2]
    feats += ["x:" + n + "_" + w for n in nums for w in words if len(w) >= 4]
    return feats


def name_analyzer(doc):
    core, compact, _addr, _anums, name_nums = doc
    return name_feats(core, compact, name_nums)


# Same feature strings hash to the same columns in both vectorizers, so one IDF serves both.
_HV = HashingVectorizer(analyzer=analyzer, n_features=N_FEATURES, alternate_sign=False,
                        norm=None, binary=True, dtype=np.float32)
_HVN = HashingVectorizer(analyzer=name_analyzer, n_features=N_FEATURES, alternate_sign=False,
                         norm=None, binary=True, dtype=np.float32)


def _hash(docs):
    return _HV.transform(docs)


def _hash_name(docs):
    return _HVN.transform(docs)


def hash_docs(df, pool, name_only=False):
    docs = list(zip(*[df[c].fill_null("").to_list() for c in COLS]))
    parts = [docs[i:i + 50000] for i in range(0, len(docs), 50000)]
    if not parts:
        return sp.csr_matrix((0, N_FEATURES), dtype=np.float32)
    return sp.vstack(pool.map(_hash_name if name_only else _hash, parts)).tocsr()


def doc_freq(mats):
    df = np.zeros(N_FEATURES, dtype=np.int64)
    for m in mats:
        df += np.bincount(m.indices, minlength=N_FEATURES)
    return df


def make_idf(df, n):
    """IDF weights; over-common (> MAX_DF) and singleton features get weight 0."""
    idf = (np.log((1 + n) / (1 + df)) + 1).astype(np.float32)
    idf[(df > MAX_DF) | (df < 2)] = 0
    return idf


def weight_(m, idf):
    """In-place IDF weighting, pruning and row L2-normalization (no matrix copies)."""
    m.data *= idf[m.indices]
    m.eliminate_zeros()
    sq = (np.add.reduceat(m.data ** 2, np.minimum(m.indptr[:-1], m.nnz - 1)) if m.nnz
          else np.zeros(m.shape[0], np.float32))
    sq[np.diff(m.indptr) == 0] = 0
    norms = np.sqrt(sq).astype(np.float32)
    norms[norms == 0] = 1
    m.data /= np.repeat(norms, np.diff(m.indptr))
    return m


def topn(A, BT, k):
    """Top-k columns of A @ BT per row -> (row, col, rank) arrays."""
    R = sp_matmul_topn(A, BT, top_n=k, threshold=0.05, sort=True, n_threads=N_JOBS).tocsr()
    rows = np.repeat(np.arange(R.shape[0]), np.diff(R.indptr))
    return rows, R.indices.astype(np.int64), (np.arange(R.nnz) - R.indptr[rows]).astype(np.int16)


def rowdot(A, B, ia, ib, chunk=1_000_000):
    """Cosine of row ia[j] of A with row ib[j] of B (rows already L2-normalized)."""
    out = np.empty(len(ia), dtype=np.float32)
    for s in range(0, len(ia), chunk):
        a, b = A[ia[s:s + chunk]], B[ib[s:s + chunk]]
        out[s:s + chunk] = np.asarray(a.multiply(b).sum(axis=1)).ravel()
    return out


def search_source(X1, X1n, X1T, Xo, Xon, k_fwd, k_name, k_rev):
    """All three channels for one (country, source) block -> DataFrame over local indices."""
    XoT = Xo.T.tocsr()
    r, c, k = topn(X1, XoT, k_fwd)
    del XoT
    fwd = pl.DataFrame({"a": r, "b": c, "rank_c": k})
    if k_name > 0:
        XonT = Xon.T.tocsr()
        r, c, k = topn(X1n, XonT, k_name)
        del XonT
        nam = pl.DataFrame({"a": r, "b": c, "rank_n": k})
    else:
        nam = pl.DataFrame({"a": [], "b": [], "rank_n": []},
                           schema={"a": pl.Int64, "b": pl.Int64, "rank_n": pl.Int16})
    r, c, k = topn(Xo, X1T, k_rev)          # rows are S2/S3 records, cols are S1 records
    rev = pl.DataFrame({"a": c, "b": r, "rank_r": k})
    u = (pl.concat([fwd.select("a", "b"), nam.select("a", "b"), rev.select("a", "b")]).unique()
         .join(fwd, on=["a", "b"], how="left").join(nam, on=["a", "b"], how="left")
         .join(rev, on=["a", "b"], how="left")
         .with_columns(pl.col("rank_c", "rank_n", "rank_r").fill_null(NOT_FOUND).cast(pl.Int16)))
    ia, ib = u["a"].to_numpy(), u["b"].to_numpy()
    return u.with_columns(score=pl.Series(rowdot(X1, Xo, ia, ib)),
                          score_n=pl.Series(rowdot(X1n, Xon, ia, ib)))


def block(split, k_fwd=K_FWD, k_name=K_NAME, k_rev=K_REV):
    t0 = time.time()
    s1 = pl.read_parquet(norm_path(split, 1), columns=["country"] + COLS).with_row_index("i1")
    results = []
    with Pool(N_JOBS) as pool:
        for country in s1["country"].unique().to_list():
            g1 = s1.filter(pl.col("country") == country)
            X1 = hash_docs(g1, pool)
            df, n = doc_freq([X1]), X1.shape[0]
            go = {}
            for s in (2, 3):   # pass 1: document frequencies only
                o = pl.read_parquet(norm_path(split, s), columns=["country"] + COLS).with_row_index("io")
                g = o.filter(pl.col("country") == country)
                go[s] = g if g.height else o   # unseen / mismatched label: search the full pool
                del o
                m = hash_docs(go[s], pool)
                df += doc_freq([m]); n += m.shape[0]
                del m
            idf = make_idf(df, n)
            del df
            X1 = weight_(X1, idf)
            X1n = weight_(hash_docs(g1, pool, name_only=True), idf)
            X1T = X1.T.tocsr()
            print(f"  [{country}] S1 {X1.shape[0]:,} nnz/row {X1.nnz / max(1, X1.shape[0]):.1f} "
                  f"({time.time() - t0:.0f}s)", flush=True)
            for s in (2, 3):   # pass 2: re-hash, weight, search, free
                Xo = weight_(hash_docs(go[s], pool), idf)
                Xon = weight_(hash_docs(go[s], pool, name_only=True), idf)
                u = search_source(X1, X1n, X1T, Xo, Xon, k_fwd, k_name, k_rev)
                del Xo, Xon
                results.append(u.select(
                    pl.Series("i1", g1["i1"].to_numpy()[u["a"].to_numpy()].astype(np.int32)),
                    pl.Series("io", go[s]["io"].to_numpy()[u["b"].to_numpy()].astype(np.int32)),
                    pl.lit(s, pl.Int8).alias("src"),
                    "score", "score_n", "rank_c", "rank_n", "rank_r"))
                print(f"    S{s} {go[s].height:,}: {u.height:,} candidates "
                      f"(fwd {int((u['rank_c'] < NOT_FOUND).sum()):,} | name-only new "
                      f"{int(((u['rank_c'] == NOT_FOUND) & (u['rank_n'] < NOT_FOUND)).sum()):,} | "
                      f"reverse-only new {int(((u['rank_c'] == NOT_FOUND) & (u['rank_n'] == NOT_FOUND)).sum()):,}) "
                      f"({time.time() - t0:.0f}s)", flush=True)
                del u
            del X1, X1n, X1T, go
    cand = pl.concat(results)
    cand.write_parquet(cand_path(split))
    print(f"total candidates {cand.height:,} ({cand.height / s1.height:.1f} per S1) in {time.time() - t0:.0f}s")
    return cand


if __name__ == "__main__":
    block(sys.argv[1])


In [ ]:
%%writefile src/blocking_eval.py
"""Blocking diagnostics on train: pair recall and oracle macro-F0.5 of the candidate union and
of each channel, plus examples of true pairs that no channel retrieved.

    python blocking_eval.py
"""
import polars as pl

from config import DATA_DIR, cand_path, norm_path

NOT_FOUND = 99


def load_truth():
    gt = pl.read_csv(DATA_DIR / "train" / "train_ground_truth.tsv", separator="\t", quote_char=None,
                     infer_schema=False).with_columns(pl.col("matched_entity_ids").fill_null(""))
    return (gt.with_columns(pl.col("matched_entity_ids").str.split(","))
            .explode("matched_entity_ids").filter(pl.col("matched_entity_ids") != "")
            .rename({"source1_entity_id": "s1", "matched_entity_ids": "o"}))


def truth_index(split="train"):
    """True pairs as (i1, io, src) row indices into the normalized parquet files."""
    ids1 = pl.read_parquet(norm_path(split, 1), columns=["entity_id"]).with_row_index("i1")
    t = load_truth().join(ids1.rename({"entity_id": "s1"}), on="s1")
    out = []
    for s in (2, 3):
        ids = pl.read_parquet(norm_path(split, s), columns=["entity_id"]).with_row_index("io")
        out.append(t.join(ids.rename({"entity_id": "o"}), on="o").with_columns(src=pl.lit(s, pl.Int8)))
    truth = pl.concat(out).select(pl.col("i1").cast(pl.Int32), pl.col("io").cast(pl.Int32), "src")
    return truth, ids1.height


def oracle(found, truth, n_s1):
    """Pair recall and macro F0.5 of a perfect matcher restricted to `found` true pairs."""
    per_true = truth.group_by("i1").len().rename({"len": "nt"})
    per_found = found.group_by("i1").len().rename({"len": "nf"})
    d = per_true.join(per_found, on="i1", how="left").with_columns(pl.col("nf").fill_null(0))
    r = d["nf"] / d["nt"]
    f = (1.25 * r / (0.25 + r)).fill_nan(0)
    n_singletons = n_s1 - d.height                     # no true matches -> empty prediction scores 1
    return found.height / truth.height, (f.sum() + n_singletons) / n_s1


def main():
    truth, n_s1 = truth_index()
    cand = pl.read_parquet(cand_path("train"))
    hit = truth.join(cand, on=["i1", "io", "src"], how="left")
    found = hit.filter(pl.col("score").is_not_null())
    print(f"true pairs {truth.height:,}; candidates {cand.height:,} ({cand.height / n_s1:.1f} per S1)")
    for name, cond in [
        ("union (all channels)", pl.lit(True)),
        ("forward K=5", pl.col("rank_c") < 5),
        ("forward K=10", pl.col("rank_c") < 10),
        ("forward + name", (pl.col("rank_c") < NOT_FOUND) | (pl.col("rank_n") < NOT_FOUND)),
        ("forward + reverse", (pl.col("rank_c") < NOT_FOUND) | (pl.col("rank_r") < NOT_FOUND)),
    ]:
        rec, f = oracle(found.filter(cond), truth, n_s1)
        n = cand.filter(cond).height
        print(f"  {name:<22} pair recall {rec:.4f} | oracle macro F0.5 {f:.4f} | cands/S1 {n / n_s1:.1f}")

    miss = hit.filter(pl.col("score").is_null()).sample(n=min(15, hit.height - found.height), seed=0)
    n1 = pl.read_parquet(norm_path("train", 1), columns=["core", "addr_n", "anums", "country"])
    no = {s: pl.read_parquet(norm_path("train", s), columns=["core", "addr_n", "anums"]) for s in (2, 3)}
    print("\ntrue pairs missed by every channel (normalized):")
    for i1, io, s in miss.select("i1", "io", "src").iter_rows():
        a, b = n1.row(i1), no[s].row(io)
        print(f"  [{a[3]}] S1: {a[0]} | {a[2]} {a[1]}\n       S{s}: {b[0]} | {b[2]} {b[1]}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/features.py
"""Stage 3: pairwise features for candidate pairs.

Feature groups (all country-agnostic, no country identity used as a feature):
  * name string similarity (rapidfuzz, vectorised C++ via cpdist)
  * legal-form agreement: same / conflicting / one side missing   <- twin detector
  * address string similarity + number-set comparison              <- twin detector
    (a twin shifts one house/unit number by a small delta; typos in true matches
     usually drop/insert digits, so both the numeric delta and digit edit distance matter)
  * record-quality flags: non-latin script, domain-style name, missing address
  * blocking context: score, rank, gap to best candidate, and the reverse view
    (how this S2/S3 record ranks among all S1 records that retrieved it)
  * record rarity: how many S1s share this exact name / address, rarest name-token frequency
  * similarity competition: this pair's name/address similarity vs. the other S1s competing for
    the same record and vs. the S1's other candidates
"""
from multiprocessing import Pool

import numpy as np
import polars as pl
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler, Levenshtein
from rapidfuzz.process import cpdist

from config import N_JOBS

NUM_FEATS = ["num_n1", "num_n2", "num_inter", "num_s1_unmatched", "num_o_unmatched", "num_first_eq",
             "num_min_delta", "num_min_digit_ed", "num_first_delta", "name_num_conflict"]


def _num_feats(args):
    a_str, b_str, na_str, nb_str = args
    A = a_str.split()[:6] if a_str else []
    B = b_str.split()[:6] if b_str else []
    sa, sb = set(A), set(B)
    inter = sa & sb
    ua, ub = [x for x in sa - inter], [x for x in sb - inter]
    min_delta, min_ed = -1.0, -1.0
    if ua and ub:
        best = None
        for x in ua:
            for y in ub:
                d = abs(int(x) - int(y)) if len(x) < 12 and len(y) < 12 else 1e9
                if best is None or d < best[0]:
                    best = (d, Levenshtein.distance(x, y))
        min_delta, min_ed = float(min(best[0], 1e6)), float(best[1])
    first_eq = -1.0 if not A or not B else float(A[0] == B[0])
    first_delta = -1.0 if not A or not B or len(A[0]) > 11 or len(B[0]) > 11 else float(min(abs(int(A[0]) - int(B[0])), 1e6))
    nna, nnb = set(na_str.split()) if na_str else set(), set(nb_str.split()) if nb_str else set()
    name_num_conflict = float(bool(nna) and bool(nnb) and not (nna & nnb))
    return (len(sa), len(sb), len(inter), len(ua), len(ub), first_eq, min_delta, min_ed, first_delta,
            name_num_conflict)


def _num_chunk(rows):
    return [_num_feats(r) for r in rows]


def _legal_feats(a, b):
    """a, b: polars Series of '|' joined legal classes."""
    la = a.str.split("|").list.eval(pl.element().filter(pl.element() != ""))
    lb = b.str.split("|").list.eval(pl.element().filter(pl.element() != ""))
    na, nb = la.list.len(), lb.list.len()
    inter = la.list.set_intersection(lb).list.len()
    both = (na > 0) & (nb > 0)
    return {
        "legal_same": (both & (inter > 0)).cast(pl.Float32),
        "legal_conflict": (both & (inter == 0)).cast(pl.Float32),
        "legal_one_missing": ((na > 0) != (nb > 0)).cast(pl.Float32),
        "legal_both_missing": ((na == 0) & (nb == 0)).cast(pl.Float32),
        "legal_exact": (a == b).cast(pl.Float32),
    }


def _sim(q, c, scorer, **kw):
    return cpdist(q, c, scorer=scorer, workers=N_JOBS, dtype=np.float32, **kw)


def _jaccard(a, b):
    ta = a.str.split(" ").list.eval(pl.element().filter(pl.element() != ""))
    tb = b.str.split(" ").list.eval(pl.element().filter(pl.element() != ""))
    inter = ta.list.set_intersection(tb).list.len()
    union = ta.list.set_union(tb).list.len()
    return (inter / union).fill_nan(0).fill_null(0).cast(pl.Float32)


def context_features(cand):
    """Blocking-context features computed over the full candidate table.
    cand: i1, io, src, score, rank."""
    return cand.with_columns(
        s1_best=pl.col("score").max().over(["i1", "src"]),
        s1_n=pl.len().over(["i1", "src"]),
        o_best=pl.col("score").max().over(["io", "src"]),
        o_n=pl.len().over(["io", "src"]),
        o_rank=pl.col("score").rank("ordinal", descending=True).over(["io", "src"]) - 1,
    ).with_columns(
        s1_gap=pl.col("s1_best") - pl.col("score"),
        o_gap=pl.col("o_best") - pl.col("score"),
        s1_second=pl.col("score").sort(descending=True).slice(1, 1).first().over(["i1", "src"]).fill_null(0),
        o_second=pl.col("score").sort(descending=True).slice(1, 1).first().over(["io", "src"]).fill_null(0),
    ).with_columns(
        o_margin=pl.when(pl.col("o_rank") == 0).then(pl.col("score") - pl.col("o_second"))
        .otherwise(pl.col("score") - pl.col("o_best")),
    ).drop("s1_second", "o_second")


RS_COLS = ["rs_nm_s1", "rs_ad_s1", "rs_nm_other", "rs_tokdf"]


def _addr_key(df):
    return pl.when(df["addr_missing"]).then(pl.lit("")).otherwise(df["anums"] + "|" + df["addr_n"])


def add_record_stats(n1, n2, n3):
    """Per-record rarity counts over the whole split (no labels involved):
      rs_nm_s1    S1 records with this exact core name   (is the name ambiguous among owners?)
      rs_ad_s1    S1 records at this exact address       (shared building / office block?)
      rs_nm_other S1 row: S2+S3 records with this name; S2/S3 row: same-source records with it
      rs_tokdf    records (all sources) containing this name's rarest core token
    A bare name with no address can only be linked safely when the name is rare; these let the
    model tell 'becerra montessori school' (unique) from 'new delhi services' (many owners)."""
    frames = [n1.select("core", ak=_addr_key(n1)), n2.select("core", ak=_addr_key(n2)),
              n3.select("core", ak=_addr_key(n3))]
    cnt = lambda s, name: (s.filter(s != "").value_counts(name="c").rename({s.name: "k"})
                           .with_columns(pl.col("c").cast(pl.Float32)).rename({"c": name}))
    nm1 = cnt(frames[0]["core"], "rs_nm_s1")
    ad1 = cnt(frames[0]["ak"], "rs_ad_s1")
    nm2, nm3 = cnt(frames[1]["core"], "n"), cnt(frames[2]["core"], "n")
    nm23 = pl.concat([nm2, nm3]).group_by("k").agg(pl.col("n").sum().alias("rs_nm_other"))
    tok = (pl.concat([f.select(pl.col("core").str.split(" ").list.unique().alias("t")) for f in frames])
           .explode("t").filter(pl.col("t") != "")["t"].value_counts(name="d").rename({"t": "k"}))

    def attach(f, other):
        f = f.with_row_index("r")
        mind = (f.select("r", t=pl.col("core").str.split(" ")).explode("t").filter(pl.col("t") != "")
                .join(tok, left_on="t", right_on="k").group_by("r").agg(rs_tokdf=pl.col("d").min().cast(pl.Float32)))
        out = (f.join(nm1, left_on="core", right_on="k", how="left")
               .join(ad1, left_on="ak", right_on="k", how="left")
               .join(other, left_on="core", right_on="k", how="left")
               .join(mind, on="r", how="left").sort("r"))
        return out.select(pl.col(RS_COLS).fill_null(0))

    return (n1.hstack(attach(frames[0], nm23)),
            n2.hstack(attach(frames[1], nm2.rename({"n": "rs_nm_other"}))),
            n3.hstack(attach(frames[2], nm3.rename({"n": "rs_nm_other"}))))


def sim_context(cand, n1, no_all, n2_len, chunk=4_000_000):
    """Similarity-aware version of the blocking-context features, computed over the FULL candidate
    table (every S1 that retrieved a record competes, whatever its fold). Blocking cosine says little
    about *which* of several same-address or same-name S1s owns a record; name/address similarity
    relative to the competing candidates does:
      o_*  : over the S1s competing for this S2/S3 record   s1_* : over this S1's own candidates
      *_js_best / *_js_gap / o_js_rank on js = name token-sort + 0.5 * full-address token-set
      *_ns_best / *_ns_gap on name token-sort; *_n90 = competitors with name token-sort >= 90."""
    fa1 = (n1["anums"] + " " + n1["addr_n"]).to_list()
    fao = (no_all["anums"] + " " + no_all["addr_n"]).to_list()
    c1, co = n1["core"].to_list(), no_all["core"].to_list()
    i1 = cand["i1"].to_numpy()
    io = cand["io"].to_numpy() + np.where(cand["src"].to_numpy() == 3, n2_len, 0)
    ns, as_ = np.empty(len(i1), np.float32), np.empty(len(i1), np.float32)
    for s in range(0, len(i1), chunk):
        a, b = i1[s:s + chunk], io[s:s + chunk]
        ns[s:s + chunk] = _sim([c1[j] for j in a], [co[j] for j in b], fuzz.token_sort_ratio)
        as_[s:s + chunk] = _sim([fa1[j] for j in a], [fao[j] for j in b], fuzz.token_set_ratio)
    del fa1, fao, c1, co
    o, s1 = ["io", "src"], ["i1", "src"]
    return cand.with_columns(c_ns=pl.Series(ns), c_fa_tset=pl.Series(as_)).with_columns(
        c_js=pl.col("c_ns") + 0.5 * pl.col("c_fa_tset"),
    ).with_columns(
        o_js_best=pl.col("c_js").max().over(o),
        o_ns_best=pl.col("c_ns").max().over(o),
        o_n90=(pl.col("c_ns") >= 90).sum().over(o).cast(pl.Float32),
        o_js_rank=(pl.col("c_js").rank("min", descending=True).over(o) - 1).cast(pl.Float32),
        s1_js_best=pl.col("c_js").max().over(s1),
        s1_ns_best=pl.col("c_ns").max().over(s1),
        s1_n90=(pl.col("c_ns") >= 90).sum().over(s1).cast(pl.Float32),
    ).with_columns(
        o_js_gap=pl.col("o_js_best") - pl.col("c_js"),
        o_ns_gap=pl.col("o_ns_best") - pl.col("c_ns"),
        s1_js_gap=pl.col("s1_js_best") - pl.col("c_js"),
        s1_ns_gap=pl.col("s1_ns_best") - pl.col("c_ns"),
    ).drop("c_ns")


def pair_features(pairs, n1, no_all, n2_len, pool=None):
    """pairs: DataFrame with i1, io, src (+ context cols). n1: normalized S1; no_all: normalized S2
    and S3 stacked (S3 rows offset by n2_len). Returns pairs with feature columns appended."""
    a = n1[pairs["i1"].to_numpy()]
    b = no_all[pairs["io"].to_numpy() + np.where(pairs["src"].to_numpy() == 3, n2_len, 0)]

    ac, bc = a["core"].to_list(), b["core"].to_list()
    acp, bcp = a["compact"].to_list(), b["compact"].to_list()
    aa, ba = a["addr_n"].to_list(), b["addr_n"].to_list()
    feats = {
        "n_ratio": _sim(ac, bc, fuzz.ratio),
        "n_tset": _sim(ac, bc, fuzz.token_set_ratio),
        "n_tsort": _sim(ac, bc, fuzz.token_sort_ratio),
        "n_partial": _sim(ac, bc, fuzz.partial_ratio),
        "n_compact_jw": _sim(acp, bcp, JaroWinkler.normalized_similarity),
        "n_compact_partial": _sim(acp, bcp, fuzz.partial_ratio),
        "n_full_ratio": _sim(a["name_n"].to_list(), b["name_n"].to_list(), fuzz.ratio),
        "a_ratio": _sim(aa, ba, fuzz.ratio),
        "a_tset": _sim(aa, ba, fuzz.token_set_ratio),
        "a_partial": _sim(aa, ba, fuzz.partial_ratio),
        "anum_ratio": _sim(a["anums"].to_list(), b["anums"].to_list(), fuzz.ratio),
    }
    out = pairs.with_columns(**{k: pl.Series(v) for k, v in feats.items()})
    out = out.with_columns(
        n_jacc=_jaccard(a["core"], b["core"]),
        a_jacc=_jaccard(a["addr_n"], b["addr_n"]),
        n_first_eq=(a["core"].str.split(" ").list.first() == b["core"].str.split(" ").list.first()).cast(pl.Float32),
        n_len1=a["core"].str.len_chars().cast(pl.Float32),
        n_len2=b["core"].str.len_chars().cast(pl.Float32),
        a_ntok1=a["addr_n"].str.count_matches(" ").cast(pl.Float32),
        a_ntok2=b["addr_n"].str.count_matches(" ").cast(pl.Float32),
        o_nonascii=b["name_nonascii"].cast(pl.Float32),
        o_domain=b["is_domain"].cast(pl.Float32),
        o_addr_missing=b["addr_missing"].cast(pl.Float32),
        s1_addr_missing=a["addr_missing"].cast(pl.Float32),
        **_legal_feats(a["legal"], b["legal"]),
        **{f"a_{c}": a[c] for c in RS_COLS},
        **{f"b_{c}": b[c] for c in RS_COLS},
    )
    rows = list(zip(a["anums"].to_list(), b["anums"].to_list(), a["name_nums"].to_list(), b["name_nums"].to_list()))
    chunks = [rows[i:i + 50000] for i in range(0, len(rows), 50000)]
    own = pool is None
    pool = pool or Pool(N_JOBS)
    num = [r for part in pool.map(_num_chunk, chunks) for r in part]
    if own:
        pool.close()
    num = np.asarray(num, dtype=np.float32).reshape(-1, len(NUM_FEATS))
    return out.with_columns(**{k: pl.Series(num[:, j]) for j, k in enumerate(NUM_FEATS)})


def feature_columns(df):
    skip = {"i1", "io", "label", "fold"}
    return [c for c in df.columns if c not in skip]


In [ ]:
%%writefile src/metrics.py
"""Macro F0.5 exactly as the challenge defines it (per S1 entity, singletons included)."""
import numpy as np
import polars as pl


def macro_f05(pred, truth, s1_ids):
    """pred/truth: DataFrames (i1, io, src) of predicted / true pairs. s1_ids: Series of all evaluated i1."""
    base = pl.DataFrame({"i1": s1_ids})
    tp = pred.join(truth, on=["i1", "io", "src"], how="semi").group_by("i1").len().rename({"len": "tp"})
    npred = pred.group_by("i1").len().rename({"len": "np"})
    ntrue = truth.group_by("i1").len().rename({"len": "nt"})
    d = base.join(tp, on="i1", how="left").join(npred, on="i1", how="left").join(ntrue, on="i1", how="left").fill_null(0)
    tp_, np_, nt_ = (d[c].to_numpy().astype(np.float64) for c in ("tp", "np", "nt"))
    p = np.divide(tp_, np_, out=np.zeros_like(tp_), where=np_ > 0)
    r = np.divide(tp_, nt_, out=np.zeros_like(tp_), where=nt_ > 0)
    denom = 0.25 * p + r
    f = np.divide(1.25 * p * r, denom, out=np.zeros_like(p), where=denom > 0)
    f[(nt_ == 0) & (np_ == 0)] = 1.0
    return float(f.mean())


In [ ]:
%%writefile src/decide.py
"""Decision rule: each S2/S3 record goes to at most one S1 entity (its highest-probability
S1 among its candidates), and only if that probability clears the threshold."""
import polars as pl


def assign(scored, threshold):
    """scored: DataFrame with i1, io, src, prob. Returns kept (i1, io, src)."""
    best = scored.with_columns(is_best=pl.col("prob") == pl.col("prob").max().over(["io", "src"]))
    return (best.filter(pl.col("is_best") & (pl.col("prob") >= threshold))
            .unique(subset=["io", "src"], keep="first").select("i1", "io", "src"))


In [ ]:
%%writefile src/model.py
"""Gradient-boosted pair classifier: LightGBM (CPU) or XGBoost (GPU when available).
Both are Apache-2.0/MIT licensed. The backend is recorded in model_meta.json."""
import numpy as np

from config import N_JOBS, SEED, WORK_DIR

LGB_PARAMS = dict(objective="binary", learning_rate=0.1, num_leaves=127, min_data_in_leaf=100,
                  feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
                  num_threads=N_JOBS, verbose=-1, seed=SEED)
XGB_PARAMS = dict(objective="binary:logistic", eval_metric="logloss", eta=0.1, max_depth=0,
                  max_leaves=127, grow_policy="lossguide", min_child_weight=5, subsample=0.8,
                  colsample_bytree=0.8, reg_lambda=1.0, tree_method="hist", max_bin=256, seed=SEED)
MAX_ROUNDS = {"lgb": 700, "xgb": 2000}   # CPU LightGBM capped for run time; GPU XGBoost can afford more
EARLY_STOP = 50


def _cuda_available():
    try:
        import subprocess
        return subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
    except OSError:
        return False


def fit(backend, train_data, Xva, yva, feats):
    """train_data: [Xtr, ytr] list; cleared as soon as the library has built its own binned copy,
    so the raw float matrix is freed before boosting starts."""
    Xtr, ytr = train_data
    if backend == "lgb":
        import lightgbm as lgb
        dtr = lgb.Dataset(Xtr, ytr, feature_name=feats, free_raw_data=True).construct()
        dva = lgb.Dataset(Xva, yva, reference=dtr).construct()
        train_data.clear()
        del Xtr, ytr
        return lgb.train(LGB_PARAMS, dtr, num_boost_round=MAX_ROUNDS["lgb"], valid_sets=[dva],
                         callbacks=[lgb.early_stopping(EARLY_STOP), lgb.log_evaluation(100)])
    import xgboost as xgb
    params = dict(XGB_PARAMS, device="cuda" if _cuda_available() else "cpu", nthread=N_JOBS)
    print("xgboost device:", params["device"], flush=True)
    dtr = xgb.QuantileDMatrix(Xtr, ytr, feature_names=feats, max_bin=params["max_bin"])
    dva = xgb.QuantileDMatrix(Xva, yva, feature_names=feats, ref=dtr)
    train_data.clear()
    del Xtr, ytr
    return xgb.train(params, dtr, num_boost_round=MAX_ROUNDS["xgb"], evals=[(dva, "valid")],
                     early_stopping_rounds=EARLY_STOP, verbose_eval=100)


def predict(backend, model, X):
    if backend == "lgb":
        return model.predict(X, num_threads=N_JOBS)
    import xgboost as xgb
    return model.predict(xgb.DMatrix(X, feature_names=model.feature_names),
                         iteration_range=(0, model.best_iteration + 1))


def save(backend, model):
    path = WORK_DIR / ("model.txt" if backend == "lgb" else "model.ubj")
    model.save_model(str(path))
    return path.name


def load(backend):
    if backend == "lgb":
        import lightgbm as lgb
        return lgb.Booster(model_file=str(WORK_DIR / "model.txt"))
    import xgboost as xgb
    m = xgb.Booster()
    m.load_model(str(WORK_DIR / "model.ubj"))
    m.set_param({"device": "cuda" if _cuda_available() else "cpu", "nthread": N_JOBS})
    return m


def importance(backend, model, feats):
    if backend == "lgb":
        return sorted(zip(model.feature_importance("gain"), feats), reverse=True)
    g = model.get_score(importance_type="total_gain")
    return sorted(((g.get(f, 0.0), f) for f in feats), reverse=True)


def best_iteration(backend, model):
    return int(model.best_iteration)


def as_matrix(df, feats):
    return np.ascontiguousarray(df.select(feats).to_numpy().astype(np.float32, copy=False))


In [ ]:
%%writefile src/train.py
"""Stage 4: train the pair classifier on train candidates and tune the decision threshold.

S1 entities are split into 5 folds by row index mod 5 (source files are shuffled). The model
trains on `--train-folds`; the threshold is tuned for macro F0.5 on `--val-fold` after the
one-owner assignment. Validation predictions are saved for error_analysis.py.

    python train.py --train-folds 0,2 --val-fold 1 --model lgb
"""
import argparse
import json
import time
from multiprocessing import Pool

import numpy as np
import polars as pl

import model as M
from blocking_eval import truth_index
from config import N_JOBS, SEED, WORK_DIR, cand_path, norm_path
from decide import assign
from features import add_record_stats, context_features, feature_columns, pair_features, sim_context
from metrics import macro_f05

NORM_COLS = ["core", "compact", "name_n", "addr_n", "anums", "name_nums", "legal",
             "name_nonascii", "is_domain", "addr_missing"]
CHUNK = 2_000_000


def load_norm(split):
    n1 = pl.read_parquet(norm_path(split, 1), columns=NORM_COLS)
    n2 = pl.read_parquet(norm_path(split, 2), columns=NORM_COLS)
    n3 = pl.read_parquet(norm_path(split, 3), columns=NORM_COLS)
    n1, n2, n3 = add_record_stats(n1, n2, n3)
    return n1, pl.concat([n2, n3]), n2.height


def load_candidates(split, n1, no_all, n2_len):
    t = time.time()
    cand = sim_context(context_features(pl.read_parquet(cand_path(split))), n1, no_all, n2_len)
    print(f"{split} candidates {cand.height:,} with context features ({time.time() - t:.0f}s)", flush=True)
    return cand


KEEP_COLS = ["i1", "io", "src", "label"]
# a few feature columns kept next to validation predictions for error_analysis.py
PROFILE_COLS = ["legal_conflict", "num_min_delta", "num_first_eq", "o_nonascii", "o_addr_missing", "n_tset"]


def mem():
    """Process RSS and machine-wide available RAM, for the log (Linux /proc; blank elsewhere)."""
    try:
        rss = int(open("/proc/self/status").read().split("VmRSS:")[1].split()[0]) / 1e6
        avail = int(open("/proc/meminfo").read().split("MemAvailable:")[1].split()[0]) / 1e6
    except (OSError, IndexError, ValueError):
        return ""
    return f"[RAM used {rss:.1f} GB, free {avail:.1f} GB]"


def featurize(pairs, n1, no_all, n2_len, pool, feats=None, keep=()):
    """Features straight into one preallocated float32 matrix, chunk by chunk, so peak memory is one
    matrix plus one chunk (not a full feature DataFrame plus its copy). Returns (X, feats, kept) where
    kept holds the `keep` columns (those present) for every row."""
    X, kept = None, []
    for i in range(0, pairs.height, CHUNK):
        part = pair_features(pairs[i:i + CHUNK], n1, no_all, n2_len, pool)
        if feats is None:
            feats = [c for c in feature_columns(part) if c != "label"]
        if X is None:
            X = np.empty((pairs.height, len(feats)), dtype=np.float32)
        X[i:i + part.height] = part.select(feats).to_numpy().astype(np.float32, copy=False)
        kept.append(part.select([c for c in keep if c in part.columns]))
        del part
        print(f"    features {min(i + CHUNK, pairs.height):,}/{pairs.height:,} {mem()}", flush=True)
    return X, feats, pl.concat(kept)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--train-folds", default="0,2")
    ap.add_argument("--val-fold", type=int, default=1)
    ap.add_argument("--sample", type=float, default=0.75, help="fraction of train-fold S1s to use (memory)")
    ap.add_argument("--val-sample", type=float, default=0.5, help="fraction of val-fold S1s to use (memory)")
    ap.add_argument("--model", choices=["lgb", "xgb"], default="lgb")
    args = ap.parse_args()
    t0 = time.time()

    truth, n_s1 = truth_index()
    # Train / validation S1 entities are chosen from ids alone, before any candidate is loaded.
    train_folds = [int(f) for f in args.train_folds.split(",")]
    all_i1 = pl.Series("i1", np.arange(n_s1, dtype=np.int32))
    tr_s1 = all_i1.filter((all_i1 % 5).is_in(train_folds))
    if args.sample < 1:
        tr_s1 = tr_s1.sample(fraction=args.sample, seed=SEED)
    val_s1 = pl.Series("i1", np.arange(args.val_fold, n_s1, 5, dtype=np.int32))
    if args.val_sample < 1:
        val_s1 = val_s1.sample(fraction=args.val_sample, seed=SEED)

    n1, no_all, n2_len = load_norm("train")
    # Context features need the FULL candidate table (every competing S1 counts), but the table is
    # ~100M rows: keep only train/val entities' rows right after, then join labels on that subset
    # (a label join on the full table made a second full copy and ran Kaggle out of memory).
    cand = load_candidates("train", n1, no_all, n2_len)
    print(f"full candidate table {cand.estimated_size() / 1e9:.1f} GB {mem()}", flush=True)
    cand = cand.filter(pl.col("i1").is_in(pl.concat([tr_s1, val_s1]).implode()))
    cand = cand.join(truth.with_columns(label=pl.lit(1, pl.Int8)), on=["i1", "io", "src"], how="left") \
               .with_columns(pl.col("label").fill_null(0))
    tr = cand.filter(pl.col("i1").is_in(tr_s1.implode()))
    va = cand.filter(pl.col("i1").is_in(val_s1.implode()))
    del cand
    print(f"train pairs {tr.height:,} (pos {tr['label'].sum():,}) | val pairs {va.height:,} ({time.time()-t0:.0f}s)")

    print(mem(), flush=True)

    with Pool(N_JOBS) as pool:
        Xtr, feats, lab = featurize(tr, n1, no_all, n2_len, pool, keep=["label"])
        ytr = lab["label"].to_numpy()
        del tr, lab
        Xva, _, va = featurize(va, n1, no_all, n2_len, pool, feats, keep=KEEP_COLS + PROFILE_COLS)
    del n1, no_all
    yva = va["label"].to_numpy()
    print(f"{len(feats)} features, train matrix {Xtr.nbytes / 1e9:.1f} GB ({time.time()-t0:.0f}s) {mem()}",
          flush=True)

    train_data = [Xtr, ytr]
    del Xtr, ytr
    model = M.fit(args.model, train_data, Xva, yva, feats)   # empties train_data once the model owns a copy
    model_file = M.save(args.model, model)
    print(f"trained ({time.time()-t0:.0f}s)", flush=True)

    va = va.with_columns(prob=pl.Series(M.predict(args.model, model, Xva)))
    del Xva
    val_truth = truth.filter(pl.col("i1").is_in(val_s1.implode()))
    best = (0, 0.5)
    for t in np.arange(0.3, 0.951, 0.025):
        f = macro_f05(assign(va, t), val_truth, val_s1)
        print(f"  thr {t:.3f}: macro F0.5 {f:.5f}")
        best = max(best, (f, float(t)))
    print(f"BEST val macro F0.5 {best[0]:.5f} at thr {best[1]:.3f} ({time.time()-t0:.0f}s)")

    va.write_parquet(WORK_DIR / "val_preds.parquet")
    pl.DataFrame({"i1": val_s1}).write_parquet(WORK_DIR / "val_s1.parquet")
    imp = M.importance(args.model, model, feats)
    print("top features:", [(f, round(g / 1e3)) for g, f in imp[:25]])
    json.dump({"backend": args.model, "model_file": model_file, "threshold": best[1], "val_f05": best[0],
               "features": feats, "best_iteration": M.best_iteration(args.model, model),
               "train_folds": train_folds, "val_fold": args.val_fold},
              open(WORK_DIR / "model_meta.json", "w"), indent=1)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/error_analysis.py
"""Where does validation macro F0.5 go? Splits errors into categories, measures how much each
category costs (F0.5 gained if that category alone were fixed), scores each country separately
and prints examples.

    python error_analysis.py          # needs work/val_preds.parquet, val_s1.parquet, model_meta.json
"""
import json

import polars as pl

from blocking_eval import truth_index
from config import WORK_DIR, norm_path
from decide import assign
from metrics import macro_f05

KEY = ["i1", "io", "src"]


def main():
    meta = json.load(open(WORK_DIR / "model_meta.json"))
    thr = meta["threshold"]
    va = pl.read_parquet(WORK_DIR / "val_preds.parquet")
    val_s1 = pl.read_parquet(WORK_DIR / "val_s1.parquet")["i1"]
    truth_all, _ = truth_index()
    truth = truth_all.filter(pl.col("i1").is_in(val_s1.implode()))
    kept = assign(va, thr)
    base = macro_f05(kept, truth, val_s1)
    print(f"val S1 {val_s1.len():,} | threshold {thr:.3f} | macro F0.5 {base:.5f}\n")

    # ---- false negatives
    fn = truth.join(kept, on=KEY, how="anti")
    in_cand = fn.join(va.select(KEY), on=KEY, how="semi")
    fn_block = fn.join(va.select(KEY), on=KEY, how="anti")
    owner = kept.select("io", "src", pl.col("i1").alias("owner"))
    fn_c = in_cand.join(owner, on=["io", "src"], how="left")
    fn_other = fn_c.filter(pl.col("owner").is_not_null()).select(KEY)
    fn_low = fn_c.filter(pl.col("owner").is_null()).select(KEY)
    # ---- false positives
    fp = kept.join(truth, on=KEY, how="anti")
    has_owner = truth_all.select("io", "src").unique()
    fp_distr = fp.join(has_owner, on=["io", "src"], how="anti")
    fp_wrong = fp.join(has_owner, on=["io", "src"], how="semi")

    cats = {
        "FN not in candidates (blocking)": ("fn", fn_block),
        "FN below threshold": ("fn", fn_low),
        "FN assigned to another S1": ("fn", fn_other),
        "FP record is a distractor": ("fp", fp_distr),
        "FP record belongs to another S1": ("fp", fp_wrong),
    }
    print(f"{'category':<34}{'pairs':>10}{'F0.5 if fixed':>16}{'gain':>9}")
    for name, (kind, df) in cats.items():
        fixed = pl.concat([kept, df]) if kind == "fn" else kept.join(df, on=KEY, how="anti")
        f = macro_f05(fixed, truth, val_s1)
        print(f"{name:<34}{df.height:>10,}{f:>16.5f}{f - base:>+9.5f}")

    # ---- what the wrong merges look like
    fpf = fp.join(va, on=KEY, how="left")
    print("\nFP profile (share of wrong merges):")
    print(fpf.select(
        legal_conflict=(pl.col("legal_conflict") == 1).mean(),
        number_shift_1_20=pl.col("num_min_delta").is_between(1, 20).mean(),
        same_first_number=(pl.col("num_first_eq") == 1).mean(),
        o_nonascii=(pl.col("o_nonascii") == 1).mean(),
        o_addr_missing=(pl.col("o_addr_missing") == 1).mean(),
        name_tset_ge_95=(pl.col("n_tset") >= 95).mean(),
        prob_median=pl.col("prob").median()))

    # ---- per country
    country = pl.read_parquet(norm_path("train", 1), columns=["country"]).with_row_index("i1") \
                .with_columns(pl.col("i1").cast(pl.Int32))
    print("\nper-country macro F0.5:")
    for c in country["country"].unique().sort().to_list():
        ids = country.filter(pl.col("country") == c).join(val_s1.to_frame(), on="i1", how="semi")["i1"]
        sub = lambda d: d.filter(pl.col("i1").is_in(ids.implode()))
        print(f"  {c:<10} n={ids.len():>8,}  F0.5 {macro_f05(sub(kept), sub(truth), ids):.5f}")

    # ---- examples
    n1 = pl.read_parquet(norm_path("train", 1), columns=["name_n", "addr_n", "anums"])
    no = {s: pl.read_parquet(norm_path("train", s), columns=["name_n", "addr_n", "anums"]) for s in (2, 3)}
    probs = va.select(*KEY, "prob")
    for name, (_, df) in cats.items():
        ex = df.join(probs, on=KEY, how="left").sample(n=min(8, df.height), seed=0)
        print(f"\n--- {name}")
        for i1, io, s, p in ex.select(*KEY, "prob").iter_rows():
            a, b = n1.row(i1), no[s].row(io)
            ptxt = "  -  " if p is None else f"{p:.3f}"
            print(f"  p={ptxt} S1: {a[0]} | {a[2]} {a[1]}\n          S{s}: {b[0]} | {b[2]} {b[1]}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/predict.py
"""Stage 5: score test candidates, apply the one-owner assignment, write both output files.

    python predict.py            # uses the model + threshold recorded in work/model_meta.json
"""
import json
import time
from multiprocessing import Pool

import numpy as np
import polars as pl

import model as M
from config import N_JOBS, OUT_DIR, WORK_DIR, norm_path
from decide import assign
from train import featurize, load_candidates, load_norm


def write_lists(path, col, s1_ids, pairs, ids2, ids3):
    """pairs: (i1, io, src) -> one row per S1 entity with comma-joined S2/S3 ids."""
    named = pl.concat([
        pairs.filter(pl.col("src") == s).join(ids.rename({"entity_id": "oid"}), on="io").select("i1", "oid")
        for s, ids in ((2, ids2), (3, ids3))
    ])
    lists = named.unique().sort("oid").group_by("i1").agg(pl.col("oid").str.join(","))
    out = (s1_ids.join(lists, on="i1", how="left")
           .select(pl.col("entity_id").alias("source1_entity_id"), pl.col("oid").fill_null("").alias(col)))
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    out.write_csv(path, separator="\t", quote_style="never")
    return out


def main():
    t0 = time.time()
    meta = json.load(open(WORK_DIR / "model_meta.json"))
    backend = meta["backend"]
    model = M.load(backend)
    n1, no_all, n2_len = load_norm("test")
    cand = load_candidates("test", n1, no_all, n2_len)
    print(f"test candidate pairs {cand.height:,} ({time.time()-t0:.0f}s)")
    probs = []
    with Pool(N_JOBS) as pool:
        for i in range(0, cand.height, 4_000_000):
            X, _, _ = featurize(cand[i:i + 4_000_000], n1, no_all, n2_len, pool, meta["features"])
            probs.append(M.predict(backend, model, X))
            del X
            print(f"  scored {min(i + 4_000_000, cand.height):,}/{cand.height:,} ({time.time()-t0:.0f}s)", flush=True)
    del n1, no_all
    cand = cand.select("i1", "io", "src").with_columns(prob=pl.Series(np.concatenate(probs)))
    kept = assign(cand, meta["threshold"])
    print(f"kept {kept.height:,} matches at threshold {meta['threshold']:.3f} ({time.time()-t0:.0f}s)")

    s1_ids = pl.read_parquet(norm_path("test", 1), columns=["entity_id"]).with_row_index("i1") \
               .with_columns(pl.col("i1").cast(pl.Int32))
    ids2 = pl.read_parquet(norm_path("test", 2), columns=["entity_id"]).with_row_index("io").with_columns(pl.col("io").cast(pl.Int32))
    ids3 = pl.read_parquet(norm_path("test", 3), columns=["entity_id"]).with_row_index("io").with_columns(pl.col("io").cast(pl.Int32))
    write_lists(OUT_DIR / "candidate_pairs.tsv", "candidate_entity_ids", s1_ids, cand, ids2, ids3)
    m = write_lists(OUT_DIR / "matching_results.tsv", "matched_entity_ids", s1_ids, kept, ids2, ids3)
    print(f"wrote {m.height:,} rows; entities with >=1 match: {(m['matched_entity_ids'] != '').sum():,} "
          f"({time.time()-t0:.0f}s)")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/translit.py
"""Indic-script name back-transliteration, learned from the training data only.

S2/S3 names for Indian businesses are often the S1 name transliterated word by word into
Devanagari, Tamil, Telugu, Gujarati, Gurmukhi, Kannada or Bengali. After `anyascii`
romanization they read like `yunik srvisej praivet limited` ("Unique Services Private Limited").

  * learned table: align romanized words with S1 words by position on training true pairs whose
    word counts agree (98% do), keep a mapping seen >= 2 times with >= 60% agreement.
  * fallback: for words the table has not seen, match a consonant skeleton against the vocabulary
    of S1 names (unlabeled records), only for skeletons of 3+ consonants to avoid short collisions.

Built once from train (ground truth + S1 names of train and test) by prepare.py and saved to
work/translit.json; applied only to records that contain Indic characters.
"""
import collections
import json
import re

INDIC_RE = re.compile(r"[ऀ-෿]")
_SKEL_RULES = [("ph", "f"), ("sh", "s"), ("ch", "c"), ("th", "t"), ("kh", "k"), ("gh", "g"), ("bh", "b"),
               ("dh", "d"), ("x", "ks"), ("q", "k"), ("c", "k"), ("w", "v"), ("z", "j")]
MIN_COUNT, MIN_SHARE, MIN_SKEL = 2, 0.6, 3


def skel(word):
    w = word.lower()
    for a, b in _SKEL_RULES:
        w = w.replace(a, b)
    w = re.sub(r"[aeiouy]", "", w)
    return re.sub(r"(.)\1+", r"\1", w)


def build(aligned_pairs, s1_vocab_counts):
    """aligned_pairs: iterable of (s1_tokens, romanized_tokens); s1_vocab_counts: Counter of S1 words."""
    cnt = collections.defaultdict(collections.Counter)
    for ta, tb in aligned_pairs:
        if len(ta) == len(tb):
            for x, y in zip(tb, ta):
                cnt[x][y] += 1
    learned = {}
    for k, c in cnt.items():
        (best, n), total = c.most_common(1)[0], sum(c.values())
        if total >= MIN_COUNT and n / total >= MIN_SHARE:
            learned[k] = best   # identity entries kept on purpose: they stop the fallback remapping them
    fallback = {}
    for w, _ in s1_vocab_counts.most_common():
        s = skel(w)
        if len(s) >= MIN_SKEL and w.isalpha():
            fallback.setdefault(s, w)
    return {"learned": learned, "fallback": fallback}


class Translit:
    def __init__(self, table):
        self.learned = table["learned"]
        self.fallback = table["fallback"]

    def map_token(self, t):
        if t in self.learned:
            return self.learned[t]
        s = skel(t)
        if len(s) >= MIN_SKEL:
            return self.fallback.get(s, t)
        return t

    def map_tokens(self, toks):
        return [self.map_token(t) for t in toks]


def save(table, path):
    json.dump(table, open(path, "w", encoding="utf-8"))


def load(path):
    return Translit(json.load(open(path, encoding="utf-8")))


## 0. Run settings and cache
`MODEL = 'xgb'` trains and predicts with XGBoost on the GPU (pick a GPU accelerator); `'lgb'` uses LightGBM on CPU.

If a previous run's `norm_*.parquet` / `cand_*.parquet` files are attached as an input dataset they are reused, skipping normalization and blocking. `KEEP_CACHE = True` leaves them in this run's output so you can create that dataset from it (Output tab → New Dataset).

In [ ]:
MODEL = 'xgb'   # 'xgb' needs a GPU accelerator; use 'lgb' on a CPU session
SAMPLE = 0.75   # share of training-fold S1s used; lower to 0.5 if training runs out of RAM
KEEP_CACHE = True
import glob, os
V = 3   # cache version: files from other normalization versions are ignored
WORK = '/kaggle/working/work'; os.makedirs(WORK, exist_ok=True)
cached = [f for pat in (f'norm_v{V}_*.parquet', f'cand_v{V}_*.parquet', f'translit_v{V}.json')
          for f in glob.glob(f'/kaggle/input/**/{pat}', recursive=True)]
for f in cached:
    dst = os.path.join(WORK, os.path.basename(f))
    if not os.path.exists(dst):
        os.symlink(f, dst)
have = lambda n: os.path.exists(os.path.join(WORK, n))
print('cached:', sorted(os.listdir(WORK)))

## 1. Learn Indic back-transliteration from train, normalize all records

In [ ]:
if not all(have(f'norm_v{V}_{sp}_s{i}.parquet') for sp in ('train', 'test') for i in (1, 2, 3)):
    run('python src/prepare.py train test')

## 2. Candidate generation (forward top-20 + reverse top-5 channels)

In [ ]:
if not have(f'cand_v{V}_train.parquet'):
    run('python src/blocking.py train')
if not have(f'cand_v{V}_test.parquet'):
    run('python src/blocking.py test')

In [ ]:
run('python src/blocking_eval.py')

## 3. Train pair classifier, tune threshold for macro F0.5, analyse errors
Rarity and similarity-competition features are computed here, so a v3 cache (same `V`) is reused.

In [ ]:
run(f'python src/train.py --train-folds 0,2 --sample {SAMPLE} --val-fold 1 --val-sample 0.5 --model {MODEL}')

In [ ]:
run('python src/error_analysis.py')

## 4. Predict test + write outputs

In [ ]:
run('python src/predict.py')

## 5. Validate with the official script

In [ ]:
import sys; sys.path.insert(0, 'src')
from config import DATA_DIR, OUT_DIR
v = DATA_DIR.parent / 'utils' / 'validate_submission.py'
run(f'python {v} --matching {OUT_DIR}/matching_results.tsv --candidate {OUT_DIR}/candidate_pairs.tsv '
    f'--test-dir {DATA_DIR}/test --check-ids')

In [ ]:
!head -5 /kaggle/working/output/matching_results.tsv
# drop symlinked inputs and validation predictions; keep norm/cand files only if KEEP_CACHE
!find /kaggle/working/work -type l -delete; rm -f /kaggle/working/work/val_*.parquet
if not KEEP_CACHE:
    !rm -f /kaggle/working/work/*.parquet
!ls -la /kaggle/working/work /kaggle/working/output